In [12]:
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START,END
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
load_dotenv()

True

In [14]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

MAIN_LLM=ChatOpenAI (
    model= 'gpt-4o-mini',
    temperature=0.3,
    api_key=OPENAI_API_KEY,
    max_retries=3
)

FALLBACK_LLM=ChatGoogleGenerativeAI(
    model= 'gemini-1.5-turbo',
    temperature=0.3,
    api_key=GEMINI_API_KEY
)

llm=MAIN_LLM.with_fallbacks([FALLBACK_LLM])

In [16]:
''' 
PS: we are making a mulitagent system, where generate agent generates the paragraphs on some topic given by user
and then the translator agent, translates the paragraphs into some other language given by the user.

our plan: one main graph, and one subgraph , we will make subgraph for the translate agent.
'''

' \nPS: we are making a mulitagent system, where generate agent generates the paragraphs on some topic given by user\nand then the translator agent, translates the paragraphs into some other language given by the user.\n\nour plan: one main graph, and one subgraph , we will make subgraph for the translate agent.\n'

In [38]:
#subgraph 
class TranslateInput(BaseModel):
    sentence :str=Field(..., description="The sentence to be translated")
    lang: str=Field(..., description="The language to translate the sentence into")
    output:str=Field(description="The translated sentence",default=None)

In [48]:
from langchain_core.prompts import ChatPromptTemplate
def TranslatorNode(State: TranslateInput):
    prompt=ChatPromptTemplate.from_template(
    f'''
    You are a Translator agent. Your only job is to translate the `input` to the `lang` language.
    `input` {State.sentence}
    `lang` {State.lang}
''')

    chain=prompt | llm
    res=chain.invoke({"input":State.sentence, "lang":State.lang})

    return {"output":res.content}

In [49]:
builder=StateGraph(TranslateInput)

builder.add_node("TranslatorNode", TranslatorNode)

builder.add_edge(START, "TranslatorNode")
builder.add_edge("TranslatorNode", END)

In [50]:
graph=builder.compile()

In [53]:
res=graph.invoke({"sentence":"Hello, you are very nice","lang":"French"})
print(res)

{'sentence': 'Hello, you are very nice', 'lang': 'French', 'output': 'Bonjour, vous êtes très gentil.'}
